In [0]:
from pyspark.sql import functions as F
import re



RAW_PATH = (
    "abfss://source@datalakegardencoffehigh"
    ".dfs.core.windows.net/coffee_db.parquet"
)

TARGET_TABLE = "high_garden.bronze.coffee_raw"




bronze_df = spark.read.parquet(RAW_PATH)

In [0]:
import re

def clean_column_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[ /-]+", "_", name)
    name = re.sub(r"[(),;{}\n\t=]", "", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

for old_name in bronze_df.columns:
    bronze_df = bronze_df.withColumnRenamed(
        old_name,
        clean_column_name(old_name)
    )

In [0]:
assert bronze_df.count() == 55
assert len(bronze_df.columns) == 33

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)